In [0]:
from pyspark.ml.recommendation import ALS
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, FloatType
from pyspark.ml.evaluation import RegressionEvaluator

# Initialize Spark session
spark = SparkSession.builder.appName("ALSRecommendation").getOrCreate()

# Define the schema for the ratings.dat file
schema = StructType([
    StructField("userId", IntegerType(), True),
    StructField("movieId", IntegerType(), True),
    StructField("rating", FloatType(), True),
    StructField("timestamp", IntegerType(), True)
])

# Load the data
data = spark.read.csv("/FileStore/tables/ratings.dat", sep="::", schema=schema)

# Split data into training and test sets
(train_data, test_data) = data.randomSplit([0.7, 0.3], seed=42)

# Initialize ALS model
als = ALS(
    maxIter=10,
    regParam=0.1,
    userCol="userId",
    itemCol="movieId",
    ratingCol="rating",
    coldStartStrategy="drop"
)

# Fit the model on the training data
model = als.fit(train_data)

# Make predictions on the test data
predictions = model.transform(test_data)

# Evaluate the model by computing the Mean Squared Error (MSE)
evaluator = RegressionEvaluator(metricName="mse", labelCol="rating", predictionCol="prediction")
mse = evaluator.evaluate(predictions)
print(f"Mean Squared Error (MSE) on test data: {mse}")


Mean Squared Error (MSE) on test data: 0.7584212672094328
